<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/19_neural_networks/19_0_PyTorch_Basics/19_0_1_Tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PyTorch Basics: Part 1
## Tensors — PyTorch's Fundamental Data Structure

---

## What This Notebook Is About

In Unit 16 you learned NumPy arrays — multi-dimensional grids of numbers that support fast vectorized operations. A PyTorch **tensor** is the same core idea with two additional superpowers that make it the foundation of every modern neural network:

1. **Device-agnostic computation** — the same code runs on a laptop CPU or a data center GPU by changing a single variable
2. **Automatic gradient tracking** — a tensor can record every operation applied to it, enabling PyTorch to compute derivatives automatically; this is how neural networks train

This notebook focuses entirely on tensors as a data structure. The goal: when you arrive at the first network-building notebook, the container holding your numbers should feel familiar — all the novelty will be in the model, not the array.

**What you will learn:**
1. How to create tensors from Python lists, NumPy arrays, and pandas DataFrames
2. How tensors compare to NumPy arrays — what is identical and where they diverge
3. What device-agnostic code looks like in practice
4. What `requires_grad=True` means — a teaser for the next notebook

In [ ]:
import numpy as np
import pandas as pd
import torch
import seaborn as sns

torch.manual_seed(42)

print(f'PyTorch version : {torch.__version__}')
print(f'NumPy version   : {np.__version__}')

---

## Section 1: What Is a Tensor?

A tensor is a multi-dimensional array of numbers. The word comes from mathematics and physics, but in PyTorch it just means: a container for numbers with a fixed shape and data type.

- A **0-dimensional tensor** (scalar) holds a single number: `5.0`
- A **1-dimensional tensor** (vector) holds a sequence: `[1.0, 2.0, 3.0]`
- A **2-dimensional tensor** (matrix) holds a grid: a table of numbers
- Higher dimensions arise naturally — a batch of 32 samples with 10 features each is a 2-D tensor of shape (32, 10)

The most common way to create a tensor is `torch.tensor()`, which works like `np.array()`.

In [ ]:
# From a Python list
a = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print('1-D tensor:', a)
print()

# From a nested list — produces a 2-D tensor
b = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print('2-D tensor:')
print(b)

Every tensor has four key attributes:

- **`.shape`** — the size along each dimension, as a `torch.Size` object (behaves like a tuple)
- **`.dtype`** — the data type of the elements (`torch.float32`, `torch.int64`, etc.)
- **`.ndim`** — the number of dimensions; same as `len(tensor.shape)`
- **`.device`** — where the tensor lives (`cpu` or `cuda:0`)

One important dtype difference from NumPy: **PyTorch defaults to `float32`**, while NumPy defaults to `float64`. Float32 is twice as fast on GPU hardware, which is why PyTorch made this choice. You will see it matter when converting between the two.

In [ ]:
print('b.shape  :', b.shape)    # torch.Size([2, 3])
print('b.dtype  :', b.dtype)    # torch.float32
print('b.ndim   :', b.ndim)     # 2
print('b.device :', b.device)   # cpu
print('b.numel():', b.numel())  # 6 — total number of elements
print()

# Compare to NumPy defaults on the same data
arr = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print('NumPy dtype default:', arr.dtype)   # float64
print('Torch dtype default:', b.dtype)     # float32

### Creating Tensors from NumPy Arrays

You will often start with data in a NumPy array (or a pandas column backed by NumPy) and need to hand it to PyTorch. There are two ways:

- **`torch.from_numpy(arr)`** — shares memory with the original array; changes to one affect the other
- **`torch.tensor(arr)`** — always copies; the safer default when you want independence

When you need to go back — for example, to pass PyTorch output to a sklearn metric function — use `.numpy()`. This only works on a CPU tensor without gradient tracking enabled.

In [ ]:
arr = np.array([1.0, 2.0, 3.0, 4.0])

# from_numpy shares memory with the original array
t_shared = torch.from_numpy(arr)
arr[0] = 99.0                          # modify the NumPy array
print('from_numpy (shared memory):')
print('  numpy array:', arr)
print('  tensor:     ', t_shared)      # the tensor changed too
print()

# tensor() always makes a copy
arr = np.array([1.0, 2.0, 3.0, 4.0])  # reset
t_copy = torch.tensor(arr)
arr[0] = 99.0
print('tensor() (always copies):')
print('  numpy array:', arr)
print('  tensor:     ', t_copy)        # unchanged
print()

# Back to NumPy (for sklearn metrics, plotting, etc.)
back = t_copy.numpy()
print('Back to NumPy:', back, '| type:', type(back).__name__)

### Creating Tensors from a pandas DataFrame

In practice, most data arrives as a pandas DataFrame. The standard pattern:
1. Do all cleaning and feature engineering in pandas (you already know how)
2. Extract the columns you need as `.values` NumPy arrays
3. Convert to tensors with `torch.tensor(..., dtype=torch.float32)`

This keeps pandas doing what it is good at — labeled, heterogeneous data — and hands PyTorch a clean numeric tensor.

We will use the Palmer Penguins dataset, which you have seen since Unit 17.

In [ ]:
penguins = sns.load_dataset('penguins').dropna().reset_index(drop=True)

# Extract two columns and convert — always specify dtype=torch.float32
flipper   = torch.tensor(penguins['flipper_length_mm'].values, dtype=torch.float32)
body_mass = torch.tensor(penguins['body_mass_g'].values,       dtype=torch.float32)

print(f'flipper   : shape={flipper.shape}, dtype={flipper.dtype}')
print(f'body_mass : shape={body_mass.shape}, dtype={body_mass.dtype}')
print()
print('First five flipper lengths (mm):', flipper[:5])
print('First five body masses   (g)  :', body_mass[:5])

### Convenience Constructors

These mirror NumPy's equivalents exactly. You use them constantly to initialize weight matrices, build test inputs, and create masks.

In [ ]:
print('torch.zeros(2, 3):')
print(torch.zeros(2, 3))
print()
print('torch.ones(2, 3):')
print(torch.ones(2, 3))
print()
print('torch.rand(2, 3):   # uniform random in [0, 1)')
print(torch.rand(2, 3))
print()
print('torch.arange(0, 10, 2):', torch.arange(0, 10, 2))    # like np.arange
print('torch.linspace(0, 1, 5):', torch.linspace(0, 1, 5))  # like np.linspace
print()
print('torch.eye(3):')
print(torch.eye(3))                                          # identity matrix

---

## Section 2: The NumPy Analogy

If you know NumPy, you already know most of the PyTorch tensor API. The two libraries were deliberately designed to be similar. The table below covers the operations you will use most often in neural network code.

| Operation | NumPy | PyTorch |
|---|---|---|
| Create from list | `np.array([1, 2, 3])` | `torch.tensor([1, 2, 3])` |
| Zeros | `np.zeros((m, n))` | `torch.zeros(m, n)` |
| Random uniform | `np.random.rand(m, n)` | `torch.rand(m, n)` |
| Index element | `arr[0, 1]` | `t[0, 1]` |
| Slice column | `arr[:, 1]` | `t[:, 1]` |
| Element-wise add | `arr + arr2` | `t + t2` |
| Element-wise multiply | `arr * arr2` | `t * t2` |
| Matrix multiply | `arr @ arr2` | `t @ t2` |
| Sum all | `arr.sum()` | `t.sum()` |
| Mean | `arr.mean()` | `t.mean()` |
| Max value | `arr.max()` | `t.max()` |
| Reshape | `arr.reshape(m, n)` | `t.reshape(m, n)` |
| Transpose | `arr.T` | `t.T` |
| Stack along new axis | `np.stack([a, b])` | `torch.stack([a, b])` |
| Concatenate | `np.concatenate([a, b])` | `torch.cat([a, b])` |

Run the code cell below to verify identical syntax on a shared example.

In [ ]:
# Same data in both libraries
data = [[1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0]]

arr = np.array(data)
t   = torch.tensor(data)

# Indexing — identical syntax
print('Indexing:')
print('  arr[0, 1] =', arr[0, 1],  '| t[0, 1] =', t[0, 1].item())
print('  arr[:, 1] =', arr[:, 1],  '| t[:, 1] =', t[:, 1])
print()

# Arithmetic
print('Element-wise multiply by 2:')
print('  NumPy:', (arr * 2).tolist())
print('  Torch:', (t   * 2).tolist())
print()

# Reductions
print('Reductions:')
print('  arr.sum()  =', arr.sum(),  '| t.sum()  =', t.sum().item())
print('  arr.mean() =', arr.mean(), '| t.mean() =', t.mean().item())
print()

# Matrix multiply: (3,3) @ (3,2) = (3,2)
W_np = np.ones((3, 2))
W_t  = torch.ones(3, 2)
print('Matrix multiply (3x3 @ 3x2):')
print('  NumPy:', (arr @ W_np).tolist())
print('  Torch:', (t   @ W_t).tolist())

### Where Tensors and Arrays Diverge

Four differences come up constantly in PyTorch code and cause bugs when forgotten.

In [ ]:
# --- Divergence 1: extracting a Python scalar ---
# NumPy: a reduction result is a numpy scalar automatically
# PyTorch: a reduction result is a 0-dimensional tensor; call .item() to get a Python number
np_sum = np.array([1.0, 2.0, 3.0]).sum()
t_sum  = torch.tensor([1.0, 2.0, 3.0]).sum()

print('1. Extracting a Python scalar:')
print('   NumPy result type :', type(np_sum).__name__)
print('   Torch result type :', type(t_sum).__name__, '<-- still a tensor')
print('   After .item()     :', type(t_sum.item()).__name__, '<-- plain Python float')
print('   Rule: use .item() whenever you need a Python number from a tensor.')
print()

# --- Divergence 2: in-place operations end with an underscore ---
x = torch.tensor([1.0, 2.0, 3.0])
x.add_(10.0)   # equivalent to x += 10 but the trailing _ marks it as in-place
print('2. In-place operations (trailing underscore):')
print('   After x.add_(10.0):', x)
print('   NumPy equivalent: arr += 10')
print()

# --- Divergence 3: default dtype (float32 vs float64) ---
print('3. Default dtype:')
print('   np.array([1.0]).dtype   :', np.array([1.0]).dtype)          # float64
print('   torch.tensor([1.0]).dtype:', torch.tensor([1.0]).dtype)     # float32
print('   Rule: always pass dtype=torch.float32 when converting from pandas/numpy.')
print()

# --- Divergence 4: .numpy() has preconditions ---
print('4. .numpy() requires CPU + no gradient:')
t_cpu = torch.tensor([1.0, 2.0, 3.0])   # CPU tensor, no grad
print('   t_cpu.numpy():', t_cpu.numpy())
print('   A tensor with requires_grad=True cannot be converted directly.')
print('   Use .detach().numpy() in that case.')

---

## Section 3: Device-Agnostic Code

One of PyTorch's design goals is that the same code should run on any hardware — a student laptop or a data center GPU — without modification. The mechanism is a single variable called `device` and the `.to(device)` method.

The standard boilerplate at the top of every PyTorch script:

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

Every tensor and every model then gets sent to `device` with `.to(device)`. The rest of the code — including the training loop — is identical whether `device` is `'cpu'` or `'cuda'`.

On this machine you will see `cpu`. That is expected — all module 19 notebooks are designed to run on CPU. Writing device-agnostic code from the start means the same notebooks work on a Colab GPU without changes.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active device : {device}')
print()
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name      : {torch.cuda.get_device_name(0)}')
else:
    print('(No GPU detected — expected for this course\'s local environment)')

In [ ]:
# Create a tensor and send it to the active device
x = torch.tensor([1.0, 2.0, 3.0])
x = x.to(device)
print(f'x.device : {x.device}')
print()

# The production pattern — these three lines appear in every training loop:
#
#   device  = 'cuda' if torch.cuda.is_available() else 'cpu'
#
#   model   = MyNetwork().to(device)   # move all model weights to device
#   X_batch = X_batch.to(device)       # move each batch of inputs
#   y_batch = y_batch.to(device)       # move each batch of labels
#
# The training loop itself is then identical on any hardware.
print('The .to(device) pattern appears in every notebook from here on.')
print('CPU is fine for this course — the pattern just keeps the code portable.')

---

## Section 4: The One Thing Tensors Have That Arrays Don't

NumPy arrays and PyTorch tensors are nearly identical data structures. But there is one feature tensors have that arrays do not: the ability to **track gradients**.

When you set `requires_grad=True` on a tensor, PyTorch begins silently recording every mathematical operation performed on it. This record — called the **computation graph** — is what allows PyTorch to automatically compute derivatives.

You do not need to understand how this works yet. The next notebook is entirely about this mechanism. Right now, just observe that the flag exists and that it changes how a tensor behaves when you operate on it.

In [ ]:
# A normal tensor — no gradient tracking
x_plain = torch.tensor([2.0, 3.0])
print('Plain tensor:')
print('  requires_grad:', x_plain.requires_grad)   # False
print('  grad_fn      :', x_plain.grad_fn)           # None
print()

# Enable gradient tracking
x_tracked = torch.tensor([2.0, 3.0], requires_grad=True)
print('Tracked tensor (requires_grad=True):')
print('  requires_grad:', x_tracked.requires_grad)   # True
print('  grad_fn      :', x_tracked.grad_fn)          # None — we created it, did not compute it
print()

# Perform operations on the tracked tensor
# y = x^2 + 3x  (applied element-wise)
y = x_tracked ** 2 + 3 * x_tracked

print('After  y = x_tracked**2 + 3*x_tracked:')
print('  y          :', y)
print('  y.grad_fn  :', y.grad_fn)    # records the last operation
print()
print('PyTorch has been silently building a record of every operation applied to x_tracked.')
print('The next notebook will show how y.backward() uses that record')
print('to compute dy/dx automatically — for every element of x, in a single pass.')

---

## Putting It All Together

| Concept | PyTorch | NumPy equivalent |
|---|---|---|
| Create from list | `torch.tensor([1, 2, 3])` | `np.array([1, 2, 3])` |
| Shape | `.shape` → `torch.Size(...)` | `.shape` → `tuple` |
| Data type | `.dtype` → `torch.float32` default | `.dtype` → `float64` default |
| Dimensions | `.ndim` | `.ndim` |
| Element count | `.numel()` | `.size` |
| Zeros / Ones | `torch.zeros(m, n)` | `np.zeros((m, n))` |
| Random uniform | `torch.rand(m, n)` | `np.random.rand(m, n)` |
| From NumPy (shared) | `torch.from_numpy(arr)` | — |
| From NumPy (copy) | `torch.tensor(arr)` | — |
| To NumPy | `.numpy()` | — |
| Extract scalar | `.item()` | automatic |
| In-place op | `.add_(val)` — trailing `_` | `arr += val` |
| Move to device | `.to(device)` | N/A |
| Track gradients | `requires_grad=True` | N/A |

**Default dtype:** PyTorch defaults to `float32`; NumPy defaults to `float64`. Always pass `dtype=torch.float32` when converting from pandas or NumPy.

**The two things that make tensors unique:**
1. `.to(device)` — moves data between CPU and GPU; the same code runs on any hardware
2. `requires_grad=True` — enables automatic gradient computation; this is how neural networks train

**What is coming next:** The `requires_grad=True` flag builds a computation graph. The next notebook makes that graph visible, shows how `.backward()` traverses it to compute derivatives automatically, and explains why this is the only practical way to train a model with millions of parameters.